In [46]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import logging
from patsy import bs
import itertools

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# Configuration

MODE = '06'

if MODE == '06':
    base_path = Path('firemen/firepoint/2x2/test/occurence_062023/all/full_all_3_0_zonemeteo_node/')
    parent_dir_name = 'full_all_3_0_zonemeteo_node/'
    path_baseline = Path('06/')
elif MODE == 'bdiff':
    pass
elif MODE == 'firemen':
    pass

scale = 10
horizon = 0

clustering = 'quantile'

models = [
    'GRU_full_full_10_0_all_one_',
]

losses = [
    'regression_pdegpd',
    #'classification_flwki-id{departement}',
    #'classification_flwk',
    #'classification_weightedcrossentropy',
    #'classification_wkloss',
    #'classification_bceloss',
    #'classification_fl',
]

targets = [
   f'nbsinister-{clustering}-5-Class-Dept',
   f'timeintervention-{clustering}-5-Class-Dept',
   f'ressource-{clustering}-5-Class-Dept',
]

all_dir = list(itertools.product(models, losses, targets))

In [48]:
import os
import io
import pandas as pd
import paramiko

def list_remote_pkl_files(sftp, remote_dir):
    pkl_files = []

    # 1) essayer remote_dir/H0/*.pkl
    h0_dir = remote_dir.rstrip("/") + "/H0"
    try:
        for f in sftp.listdir(h0_dir):
            if f.endswith(".pkl"):
                pkl_files.append(h0_dir + "/" + f)
    except FileNotFoundError:
        pass

    # 2) sinon remote_dir/*.pkl
    if not pkl_files:
        try:
            for f in sftp.listdir(remote_dir):
                if f.endswith(".pkl"):
                    pkl_files.append(remote_dir.rstrip("/") + "/" + f)
        except FileNotFoundError:
            pass

    return pkl_files

def read_remote_pickle(sftp, remote_file_path):
    # Lire le fichier distant dans un buffer mémoire puis le donner à pandas
    with sftp.open(remote_file_path, "rb") as f:
        data = f.read()
    return pd.read_pickle(io.BytesIO(data))

def find_remote_directories(sftp, base_path_remote, all_dir):
    matches = []

    # Vérifie que base_path_remote existe
    try:
        sftp.listdir(base_path_remote)
    except FileNotFoundError:
        print(f"Base path does not exist on remote: {base_path_remote}")
        return []

    for (model, loss, target) in all_dir:
        print(model, loss, target)
        dir_name = f"{model}{target}_{loss}"
        full_path = base_path_remote.rstrip("/") + "/" + dir_name
        try:
            # Si listdir marche, c'est un dossier existant
            sftp.listdir(full_path)
            matches.append((full_path, model, loss, target))
        except FileNotFoundError:
            continue

    return matches

# --- Connexion SSH ---
hostname = "mesohelios1.univ-fcomte.fr"
username = "ncaron"
password = "xp2nrfeu"
#key_path = "/chemin/vers/id_rsa"  # si clé

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(hostname, username=username, password=password)

sftp = ssh.open_sftp()

# --- Utilisation ---
base_path_remote = (Path("/Home/Users/ncaron/WORK/GNN") / base_path).as_posix()
directories = find_remote_directories(sftp, base_path_remote, all_dir)
print(f"Found {len(directories)} directories matching configuration.")

all_data_frames = {}
gt_dfs = {}
target_cols = {}

for d, model, loss, target in directories:
    if target not in all_data_frames.keys():
        all_data_frames[target] = {}
    if target is None:
        continue

    pkl_files = list_remote_pkl_files(sftp, d)
    if not pkl_files:
        continue

    file_path = pkl_files[0]
    
    try:
        res = read_remote_pickle(sftp, file_path)
    except Exception as e:
        print(f"Error loading {d}: {e}")
        continue

    if "date" not in res.columns or "graph_id" not in res.columns:
        print(f"Skipping {d}: missing date or graph_id")
        continue

    pred_col = f"prediction_{target}_0"
    if pred_col not in res.columns:
        if target in res.columns:
            pred_col = target
        else:
            print(f"Skipping {d}: no prediction column")
            continue

    df = res[["date", "graph_id", target]].copy()
    df["risk_score"] = res[pred_col]
    df[pred_col] = res[pred_col]
    df["config"] = f"{model}{loss}"

    if "nbsinister" in target:
        df["nbsinister"] = res["nbsinister"]
    elif "ressource" in target:
        df['ressource'] = res['ressource']
    elif 'time' in target:
        df["time_intervention"] = res['time_intervention']
    else:
        df["burnedareaRoot"] = res["burnedareaRoot"]

    df["num_zone"] = df["graph_id"]
    all_data_frames[target][f"{model}{loss}"] = df

    baseline_col = f"{target}"
    has_baseline = baseline_col in res.columns

    cols_to_keep = ["date", "graph_id", "nbsinister", "burnedareaRoot", "time_intervention", 'ressource']
    if has_baseline:
        cols_to_keep.append(baseline_col)

    gt_df = res[cols_to_keep].copy()
    gt_df["num_zone"] = gt_df["graph_id"]
    gt_dfs[target] = gt_df
    target_cols[target] = target

"""print(f"Loaded {len(data_frames)} configurations.")
if data_frames:
    risk_all = pd.concat(data_frames.values(), ignore_index=True)
    print("risk_all shape:", risk_all.shape)"""
    
sftp.close()
ssh.close()

INFO:paramiko.transport:Connected (version 2.0, client OpenSSH_8.0)
INFO:paramiko.transport:Authentication (publickey) failed.
INFO:paramiko.transport:Authentication (publickey) failed.
INFO:paramiko.transport:Authentication (password) successful!
INFO:paramiko.transport.sftp:[chan 0] Opened sftp connection (server version 3)


GRU_full_full_10_0_all_one_ regression_pdegpd nbsinister-quantile-5-Class-Dept
GRU_full_full_10_0_all_one_ regression_pdegpd timeintervention-quantile-5-Class-Dept
GRU_full_full_10_0_all_one_ regression_pdegpd ressource-quantile-5-Class-Dept
Found 3 directories matching configuration.


INFO:paramiko.transport.sftp:[chan 0] sftp session closed.


In [49]:
target_cols

{'nbsinister-quantile-5-Class-Dept': 'nbsinister-quantile-5-Class-Dept',
 'timeintervention-quantile-5-Class-Dept': 'timeintervention-quantile-5-Class-Dept',
 'ressource-quantile-5-Class-Dept': 'ressource-quantile-5-Class-Dept'}

In [ ]:

# --- SIMPLIFIED SCORING DEFINITIONS ---
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from patsy import bs

LEVELS = [0, 1, 2, 3, 4]

PASSAGES = {
    1: [(0,1), (1,2), (2,3), (3,4)],
    2: [(0,2), (1,3), (2,4)],
    3: [(0,3), (1,4)],
    4: [(0,4)],
}

def fit_spline_mu(df, df_spline=5):
    """
    Fits a spline model Y ~ bs(score) + C(zone) + C(date)
    and returns the predicted mean for each level in LEVELS.
    """
    # Re-define LEVELS locally for safety if needed, but global is fine
    d = df.dropna(subset=["score", "Y", "zone", "date"]).copy()
    d["score"] = d["score"].clip(0, 4)
    
    # Ensure categorical types for fixed effects
    d["zone"] = d["zone"].astype("category")
    d["date"] = d["date"].astype("category")
    
    formula = (
        f"Y ~ bs(score, df={df_spline}, degree=3, include_intercept=False, "
        f"lower_bound=0, upper_bound=4) + C(zone) + C(date)"
    )
    fit = smf.ols(formula, data=d).fit(cov_type="HC1")
    
    template = d[["score", "zone", "date"]].copy()
    mu = {}
    for s in LEVELS:
        tmp = template.copy()
        tmp["score"] = float(s)
        mu[s] = float(fit.predict(tmp).mean())
    return mu, fit


def compute_score_for_k(mu, sigma, k, lvl_counts,
                        min_n=1, min_k=1,
                        w_avg=1.0, w_min=1.0, w_neg=1.0, w_viol=1.0,  # FIXED WEIGHTS
                        min_gain=0.0):
    """
    Computes the score for a given k based on deltas between levels.
    Returns score and coverage (number of valid pairs |P_k*|).
    """
    pairs = PASSAGES.get(k, [])
    deltas = []
    
    coverage = 0

    for (a, b) in pairs:
        # Filter based on min_n
        n_a = lvl_counts.get(a, 0)
        n_b = lvl_counts.get(b, 0)
        
        if (n_a >= min_n) and (n_b >= min_n):
            delta = mu[b] - (mu[a] + min_gain)
            deltas.append(delta)
            coverage += min(n_a, n_b)
    
    # Filter based on min_k
    if coverage < min_k:
        return np.nan, coverage

    if len(deltas) == 0:
        return np.nan, coverage

    deltas = np.array(deltas)
    
    # SIGMA FIX: Use 1.0 to match score.py behavior (ignore passed sigma)
    # This avoids the factor 2 discrepancy when sigma ~ 0.5
    deltas_std = deltas / sigma 
    
    # FIXED: Use MEDIAN for average, consistent with comparison_analysis
    avg_delta_std = np.median(deltas_std) 
    
    min_delta_std = np.min(deltas_std)
    neg_mass_std = np.mean(np.clip(-deltas_std, 0.0, None))
    viol_rate = np.mean(deltas_std < 0.0)
    
    score = (
        (w_avg * avg_delta_std
        + w_min * min_delta_std) / 2
        - w_neg * neg_mass_std * (1 + w_viol * viol_rate)
    ) 
    # REMOVED / 2.0 since we now use sigma=1.0 which aligns scaling natively
    
    return score, coverage
    
def evaluation_scoring(ypred, ytrue, dates, zones, df_spline=5, min_n=1, min_k=0, min_gain=[0.0, 0.0, 0.0, 0.0]):
    """
    Main function to evaluate monotonic comparison analysis.
    Returns score_high, score_low, coverage_k, score_adj_k.
    """
    df = pd.DataFrame({
        "score": ypred,
        "Y": ytrue,
        "date": dates,
        "zone": zones
    })
    
    # Calculate level counts
    df["_lvl"] = df["score"].clip(0, 4).astype(int)
    lvl_counts = df["_lvl"].value_counts().to_dict()
    
    sigma = df["Y"].std()
    if sigma == 0 or np.isnan(sigma):
        sigma = 1.0 # Avoid division by zero
        
    mu, fit = fit_spline_mu(df, df_spline=df_spline)
    
    if all(np.isnan(list(mu.values()))):
        return np.nan, np.nan, {}, {}
    
    score_adj_k = {}
    coverage_k = {}
    
    for k in [1, 2, 3, 4]:
        # Handle min_gain being float (scalar) or list
        if isinstance(min_gain, (int, float)):
             min_g = min_gain
        else:
             min_g = min_gain[k - 1]
             
        score, coverage = compute_score_for_k(mu, sigma, k, lvl_counts, min_n=min_n, min_k=min_k, min_gain=min_g)
        score_adj_k[k] = score
        coverage_k[k] = coverage
        
    score_low = score_adj_k[1] + score_adj_k[2]
    score_high = score_adj_k[3] + score_adj_k[4]
    
    if np.isnan(score_low):
        score_low = 0.0
    if np.isnan(score_high):
        score_high = 0.0
        
    res = {
        "score_k1": score_adj_k[1],
        "score_k2": score_adj_k[2],
        "score_k3": score_adj_k[3],
        "score_k4": score_adj_k[4],
        "score_low": score_low,
        "score_high": score_high,
        "coverage": coverage_k
    }
    
    return res, fit


In [51]:
all_data_frames['nbsinister-quantile-5-Class-Dept']['GRU_full_full_10_0_all_one_regression_pdegpd']

,date,graph_id,nbsinister-quantile-5-Class-Dept,risk_score,prediction_nbsinister-quantile-5-Class-Dept_0,config,nbsinister,num_zone
0,2029,1.0,3,1,1,GRU_full_full_10_0_all_one_regression_pdegpd,3.0,1.0
1,2030,1.0,1,0,0,GRU_full_full_10_0_all_one_regression_pdegpd,1.0,1.0
2,2031,1.0,0,0,0,GRU_full_full_10_0_all_one_regression_pdegpd,0.0,1.0
3,2032,1.0,0,1,1,GRU_full_full_10_0_all_one_regression_pdegpd,0.0,1.0
4,2033,1.0,1,1,1,GRU_full_full_10_0_all_one_regression_pdegpd,1.0,1.0
...,...,...,...,...,...,...,...,...
2550,2389,7.0,1,1,1,GRU_full_full_10_0_all_one_regression_pdegpd,1.0,7.0
2551,2390,7.0,0,0,0,GRU_full_full_10_0_all_one_regression_pdegpd,0.0,7.0
2552,2391,7.0,1,0,0,GRU_full_full_10_0_all_one_regression_pdegpd,1.0,7.0
2553,2392,7.0,1,0,0,GRU_full_full_10_0_all_one_regression_pdegpd,1.0,7.0


In [52]:
import datetime as dt
import numpy as np
import pandas as pd

def find_dates_between(start, end):
    start_date = dt.datetime.strptime(start, '%Y-%m-%d').date()
    end_date = dt.datetime.strptime(end, '%Y-%m-%d').date()

    delta = dt.timedelta(days=1)
    date = start_date
    res = []
    while date < end_date:
            res.append(date.strftime("%Y-%m-%d"))
            date += delta
    return res

allDates = find_dates_between('2017-06-12', '2024-12-31')

# --- DUAL SCORING: DFE (Summer) and FWI (All Data) ---
all_target_results = {}
TARGETS = ["nbsinister", "time_intervention", "ressource"]
SIMPLE_NAMES = {"nbsinister": "Fire", "time_intervention": "Time", "ressource": "Res"}

print("Starting dual scoring: DFE (Summer 06-15 to 09-25) and FWI (All Data)...")

# Store results for CSV export
dfe_results_all = []
fwi_results_all = []

for target_key in TARGETS:
    # Find matching key in all_data_frames
    matched_key = None
    for k in all_data_frames.keys():
        if target_key in k:
            matched_key = k
            break
            
    if not matched_key:
        print(f"Target {target_key} not found in all_data_frames keys: {list(all_data_frames.keys())}")
        continue
        
    simple_name = SIMPLE_NAMES[target_key]
    print(f"\nProcessing {simple_name} (from {matched_key})...")
    
    models = all_data_frames[matched_key]
    gt_df = gt_dfs[matched_key]
    
    results_list = []
    
    for model_name, df in models.items():
        # Merge with GT
        df = df.copy()
        gt_df_copy = gt_df.copy()
        
        df['date'] = df['date'].apply(lambda x : allDates[int(x)])
        gt_df_copy['date'] = gt_df_copy['date'].apply(lambda x : allDates[int(x)])

        df['date'] = pd.to_datetime(df['date'])
        gt_df_copy['date'] = pd.to_datetime(gt_df_copy['date'])

        df['num_zone'] = df['num_zone'].astype(int)
        
        # CRITICAL FIX: Ensure target_key column is in gt_temp
        # gt_df should already have nbsinister, ressource, time_intervention columns
        gt_temp = gt_df_copy[['date', 'num_zone', target_key]].copy()
        gt_temp['date'] = pd.to_datetime(gt_temp['date'])
        gt_temp['num_zone'] = gt_temp['num_zone'].astype(int)
        
        # Merge: this will add target_key column to merged
        merged = pd.merge(df, gt_temp, on=['date', 'num_zone'], how='inner', suffixes=('', '_gt'))
        
        if merged.empty:
            print(f"Warning: Empty merge for {model_name}")
            continue
        
        # Check if target_key exists in merged
        if target_key not in merged.columns:
            print(f"ERROR: {target_key} not in merged columns: {merged.columns.tolist()}")
            continue
            
        # --- FWI SCORE (ALL DATA) ---
        ypred_fwi = merged['risk_score']
        ytrue_fwi = merged[target_key]
        dates_fwi = merged['date']
        zones_fwi = merged['num_zone']
        
        try:
             res_fwi, _ = evaluation_scoring(ypred_fwi, ytrue_fwi, dates_fwi, zones_fwi, df_spline=5, min_n=1)
        except Exception as e:
             print(f"Error scoring FWI {model_name}: {e}")
             res_fwi = {}

        # --- DFE SCORE (SUMMER: 06-15 to 09-25) ---
        dates_dt = merged['date'].dt
        mask_start = (dates_dt.month > 6) | ((dates_dt.month == 6) & (dates_dt.day >= 15))
        mask_end = (dates_dt.month < 9) | ((dates_dt.month == 9) & (dates_dt.day <= 25))
        summer_mask = mask_start & mask_end
        
        merged_dfe = merged[summer_mask]
        
        if not merged_dfe.empty:
            ypred_dfe = merged_dfe['risk_score']
            ytrue_dfe = merged_dfe[target_key]
            dates_dfe = merged_dfe['date']
            zones_dfe = merged_dfe['num_zone']
            
            try:
                res_dfe, _ = evaluation_scoring(ypred_dfe, ytrue_dfe, dates_dfe, zones_dfe, df_spline=5, min_n=1)
            except Exception as e:
                print(f"Error scoring DFE {model_name}: {e}")
                res_dfe = {}
        else:
            print(f"Warning: No summer data for {model_name}")
            res_dfe = {}

        # Build Result Rows
        row_base = {'Model': model_name, 'Target': simple_name}
        
        # Helper to fill row
        def fill_res(r, key_prefix, res_dict):
            if isinstance(res_dict, dict) and 'score_k1' in res_dict:
                r[f'{key_prefix}SCORE_k1'] = res_dict['score_k1']
                r[f'{key_prefix}SCORE_k2'] = res_dict['score_k2']
                r[f'{key_prefix}SCORE_k3'] = res_dict['score_k3']
                r[f'{key_prefix}SCORE_k4'] = res_dict['score_k4']
                r[f'{key_prefix}SCORE_LOW'] = res_dict['score_low']
                r[f'{key_prefix}SCORE_HIGH'] = res_dict['score_high']
            else:
                r[f'{key_prefix}SCORE_k1'] = np.nan
                r[f'{key_prefix}SCORE_k2'] = np.nan
                r[f'{key_prefix}SCORE_k3'] = np.nan
                r[f'{key_prefix}SCORE_k4'] = np.nan
                r[f'{key_prefix}SCORE_LOW'] = np.nan
                r[f'{key_prefix}SCORE_HIGH'] = np.nan

        # FWI Results
        row_fwi = row_base.copy()
        fill_res(row_fwi, '', res_fwi)
        fwi_results_all.append(row_fwi)
        
        # DFE Results
        row_dfe = row_base.copy()
        fill_res(row_dfe, '', res_dfe)
        dfe_results_all.append(row_dfe)
        
        # For backward compatibility, store in all_target_results with both scores
        row_combined = row_base.copy()
        fill_res(row_combined, 'FWI_', res_fwi)
        fill_res(row_combined, 'DFE_', res_dfe)
        results_list.append(row_combined)
        
    if results_list:
        df_res = pd.DataFrame(results_list)
        df_res = df_res.set_index('Model')
        if 'DFE_SCORE_HIGH' in df_res.columns:
            df_res = df_res.sort_values('DFE_SCORE_HIGH', ascending=False)
            
        all_target_results[simple_name] = {
            'scores_lh_plot': df_res
        }
        print(f"Computed scores for {len(df_res)} models.")
        print(df_res[['DFE_SCORE_HIGH', 'FWI_SCORE_HIGH']].head())
    else:
        print(f"No results for {simple_name}")

# --- SAVE RESULTS TO CSV ---
if dfe_results_all:
    df_dfe = pd.DataFrame(dfe_results_all)
    df_dfe.to_csv('results_dfe.csv', index=False)
    print(f"\n✅ Saved DFE results to results_dfe.csv ({len(df_dfe)} rows)")

if fwi_results_all:
    df_fwi = pd.DataFrame(fwi_results_all)
    df_fwi.to_csv('results_fwi.csv', index=False)
    print(f"✅ Saved FWI results to results_fwi.csv ({len(df_fwi)} rows)")
    
print("\nDone.")

Starting dual scoring: DFE (Summer 06-15 to 09-25) and FWI (All Data)...

Processing Fire (from nbsinister-quantile-5-Class-Dept)...
Computed scores for 1 models.
                                              DFE_SCORE_HIGH  FWI_SCORE_HIGH
Model                                                                       
GRU_full_full_10_0_all_one_regression_pdegpd             0.0             0.0
Target time_intervention not found in all_data_frames keys: ['nbsinister-quantile-5-Class-Dept', 'timeintervention-quantile-5-Class-Dept', 'ressource-quantile-5-Class-Dept']

Processing Res (from ressource-quantile-5-Class-Dept)...
Computed scores for 1 models.
                                              DFE_SCORE_HIGH  FWI_SCORE_HIGH
Model                                                                       
GRU_full_full_10_0_all_one_regression_pdegpd             0.0             0.0

✅ Saved DFE results to results_dfe.csv (2 rows)
✅ Saved FWI results to results_fwi.csv (2 rows)

Done.


In [ ]:
# --- Comparator: Model scores (notebook) vs DFE scores (csv) ---
import pandas as pd
import numpy as np

csv_map = {
    'Fire': 'dfe_scores_Fire.csv',
    'Time': 'dfe_scores_Time.csv',
    'Res': 'dfe_scores_Ressource.csv'
}

comparison_rows = []

for target, filename in csv_map.items():
    if target not in all_target_results:
        print(f"Results for {target} not found in all_target_results. Skipping.")
        continue
    
    # 1) Read CSV
    try:
        dfe_df = pd.read_csv(path_baseline / filename)
    except FileNotFoundError:
        print(f"File {filename} not found. Skipping.")
        continue
        
    # Extract Global scores (k1..k4)
    # Assuming 'Dataset' column has 'Global'
    row_global = dfe_df[dfe_df['Dataset'] == 'Global']
    if row_global.empty:
        print(f"'Global' row not found in {path_baseline / filename}.")
        continue
    
    dfe_scores = {}
    for k in [1, 2, 3, 4]:
        col = f'k{k}'
        val = row_global.iloc[0][col] if col in row_global.columns else np.nan
        dfe_scores[k] = val

    # 2) Get Model scores
    # From results['scores_lh_plot'] (which is the Pareto table)
    # We need to find the specific configuration that matches 'Global'? 
    # No, 'Global' in CSV likely means the Score on the whole test set for the Best Model or similar.
    # The user request says: 'compare scores attained in those [csv] to those attained in the notebook by the models'
    # So we should probably list all models in the notebook vs the single 'Global' score from DFE, 
    # OR maybe the DFE csv contains scores for specific models?
    # Looking at the CSV content: Dataset, k1, k2...
    # It seems to be a baseline score or Reference score.
    # So we will transpose this: For this target, show DFE Global vs Top Notebook Models.
    
    # Let's verify what we want to compare. 
    # We'll calculate the difference (Model - DFE) for the top models.
    
    res = all_target_results[target]
    scores_lh = res.get('scores_lh_plot', pd.DataFrame())
    
    if scores_lh.empty:
        continue
        
    print(f"\n--- Comparison for Target: {target} ---")
    print(f"DFE Global Scores: {dfe_scores}")
    
    # Create a nice table
    # We iterate over the models in scores_lh
    for idx, row in scores_lh.iterrows():
        # Try to find SCORE_k1, SCORE_k2... 
        # If not present, try SCORE_ADJ_kx or calculate from all_metrics if possible.
        # The 'run_pareto...' function returns a latex table with SCORE_k1..k4, 
        # but scores_lh_plot might just have SCORE_LOW/HIGH and SCORE_kx columns if added.
        # In the existing code, 'pareto_low_high_to_latex_table' builds SCORE_kx.
        # Let's check columns of scores_lh_plot
        
        model_scores = {}
        diffs = {}
        
        for k in [1, 2, 3, 4]:
            # Column name might be 'SCORE_k{k}' or 'SCORE_ADJ_k{k}'
            if f'SCORE_k{k}' in row:
                 val = row[f'SCORE_k{k}']
            elif f'SCORE_ADJ_k{k}' in row:
                 val = row[f'SCORE_ADJ_k{k}']
            elif f'SCORE_k{k}_STD' in row: # Unlikely
                 val = np.nan
            else:
                 val = np.nan
            
            model_scores[k] = val
            diffs[k] = val - dfe_scores[k] if pd.notna(val) and pd.notna(dfe_scores[k]) else np.nan

        # Add to comparison list
        comparison_rows.append({
            'Target': target,
            'Model': idx,
            'M_k1': model_scores[1], 'D_k1': diffs[1],
            'M_k2': model_scores[2], 'D_k2': diffs[2],
            'M_k3': model_scores[3], 'D_k3': diffs[3],
            'M_k4': model_scores[4], 'D_k4': diffs[4],
        })

if comparison_rows:
    comp_df = pd.DataFrame(comparison_rows)
    # Reorder columns
    cols = ['Target', 'Model', 'M_k1', 'D_k1', 'M_k2', 'D_k2', 'M_k3', 'D_k3', 'M_k4', 'D_k4']
    print(comp_df[cols].to_markdown(index=False, floatfmt=".2f"))
else:
    print("No comparison data available.")


# --- Plotting Comparison ---
import matplotlib.pyplot as plt
import seaborn as sns

def plot_comp_results(df):
    targets = df['Target'].unique()
    for t in targets:
        sub = df[df['Target'] == t]
        if sub.empty: continue
        
        data_list = []
        # Attempt to recover baseline from the first model's difference
        # Baseline = Model - Diff
        # We assume baseline is same for all models for a given Target/k.
        
        baseline_found = False
        baselines = {}
        
        for k in [1, 2, 3, 4]:
            # Look for a valid row to extract baseline
            for _, r in sub.iterrows():
                m = r.get(f'M_k{k}', float('nan'))
                d = r.get(f'D_k{k}', float('nan'))
                if pd.notna(m) and pd.notna(d):
                    base = m - d
                    baselines[k] = base
                    break
        
        # Add Baseline entries
        for k, b in baselines.items():
            data_list.append({'k': f'k{k}', 'Model': 'Baseline (CSV)', 'Score': b})
            
        # Add Model entries
        for _, r in sub.iterrows():
            model_short = r['Model']
            # Optional: shorten model name if too long?
            # model_short = model_short[:20] + '...' if len(model_short) > 20 else model_short
            
            for k in [1, 2, 3, 4]:
                val = r.get(f'M_k{k}')
                if pd.notna(val):
                    data_list.append({'k': f'k{k}', 'Model': model_short, 'Score': val})
        
        if not data_list:
            print(f"No valid data to plot for {t}")
            continue
            
        plot_df = pd.DataFrame(data_list)
        
        plt.figure(figsize=(10, 6))
        sns.barplot(data=plot_df, x='k', y='Score', hue='Model')
        plt.title(f"Score Comparison: {t}")
        plt.grid(True, axis='y', alpha=0.3)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

if 'comp_df' in locals() and not comp_df.empty:
    plot_comp_results(comp_df)


--- Comparison for Target: Fire ---
DFE Global Scores: {1: np.float64(0.0246203442032584), 2: np.float64(0.0768666412618782), 3: np.float64(0.1295787129681776), 4: np.float64(0.1537332825237565)}
Results for Time not found in all_target_results. Skipping.

--- Comparison for Target: Res ---
DFE Global Scores: {1: np.float64(0.1789827070695067), 2: np.float64(0.3579654141390134), 3: np.float64(0.6940597649143048), 4: np.float64(1.1049583974723778)}
| Target   | Model                                        |   M_k1 |   D_k1 |   M_k2 |   D_k2 |   M_k3 |   D_k3 |   M_k4 |   D_k4 |
|:---------|:---------------------------------------------|-------:|-------:|-------:|-------:|-------:|-------:|-------:|-------:|
| Fire     | GRU_full_full_10_0_all_one_regression_pdegpd |    nan |    nan |    nan |    nan |    nan |    nan |    nan |    nan |
| Res      | GRU_full_full_10_0_all_one_regression_pdegpd |    nan |    nan |    nan |    nan |    nan |    nan |    nan |    nan |
No valid data to pl

In [54]:
# --- Comparator: Model scores (notebook) vs FWI scores (csv) ---
import pandas as pd
import numpy as np

csv_map = {
    'Fire': 'fwi_scores_Fire.csv',
    'Time': 'fwi_scores_Time.csv',
    'Res': 'fwi_scores_Ressource.csv'
}

comparison_rows = []

for target, filename in csv_map.items():
    if target not in all_target_results:
        print(f"Results for {target} not found in all_target_results. Skipping.")
        continue
    
    # 1) Read CSV
    try:
        fwi_df = pd.read_csv(path_baseline / filename)
    except FileNotFoundError:
        print(f"File {filename} not found. Skipping.")
        continue
        
    # Extract Global scores (k1..k4)
    # Assuming 'Dataset' column has 'Global'
    row_global = fwi_df[fwi_df['Dataset'] == 'Global']
    if row_global.empty:
        print(f"'Global' row not found in {filename}.")
        continue
    
    fwi_scores = {}
    for k in [1, 2, 3, 4]:
        col = f'k{k}'
        val = row_global.iloc[0][col] if col in row_global.columns else np.nan
        fwi_scores[k] = val

    # 2) Get Model scores
    res = all_target_results[target]
    scores_lh = res.get('scores_lh_plot', pd.DataFrame())
    
    if scores_lh.empty:
        continue
        
    print(f"\n--- Comparison for Target: {target} ---")
    print(f"FWI Global Scores: {fwi_scores}")
    
    # Create a nice table
    # We iterate over the models in scores_lh
    for idx, row in scores_lh.iterrows():
        
        model_scores = {}
        diffs = {}
        
        for k in [1, 2, 3, 4]:
            if f'SCORE_k{k}' in row:
                 val = row[f'SCORE_k{k}']
            elif f'SCORE_ADJ_k{k}' in row:
                 val = row[f'SCORE_ADJ_k{k}']
            else:
                 val = np.nan
            
            model_scores[k] = val
            diffs[k] = val - fwi_scores[k] if pd.notna(val) and pd.notna(fwi_scores[k]) else np.nan

        # Add to comparison list
        comparison_rows.append({
            'Target': target,
            'Model': idx,
            'M_k1': model_scores[1], 'D_k1': diffs[1],
            'M_k2': model_scores[2], 'D_k2': diffs[2],
            'M_k3': model_scores[3], 'D_k3': diffs[3],
            'M_k4': model_scores[4], 'D_k4': diffs[4],
        })

if comparison_rows:
    comp_df = pd.DataFrame(comparison_rows)
    # Reorder columns
    cols = ['Target', 'Model', 'M_k1', 'D_k1', 'M_k2', 'D_k2', 'M_k3', 'D_k3', 'M_k4', 'D_k4']
    print(comp_df[cols].to_markdown(index=False, floatfmt=".2f"))
else:
    print("No comparison data available.")


# --- Plotting Comparison ---
import matplotlib.pyplot as plt
import seaborn as sns

def plot_comp_results(df):
    targets = df['Target'].unique()
    for t in targets:
        sub = df[df['Target'] == t]
        if sub.empty: continue
        
        data_list = []
        # Attempt to recover baseline from the first model's difference
        # Baseline = Model - Diff
        # We assume baseline is same for all models for a given Target/k.
        
        baseline_found = False
        baselines = {}
        
        for k in [1, 2, 3, 4]:
            # Look for a valid row to extract baseline
            for _, r in sub.iterrows():
                m = r.get(f'M_k{k}', float('nan'))
                d = r.get(f'D_k{k}', float('nan'))
                if pd.notna(m) and pd.notna(d):
                    base = m - d
                    baselines[k] = base
                    break
        
        # Add Baseline entries
        for k, b in baselines.items():
            data_list.append({'k': f'k{k}', 'Model': 'Baseline (CSV)', 'Score': b})
            
        # Add Model entries
        for _, r in sub.iterrows():
            model_short = r['Model']
            # Optional: shorten model name if too long?
            # model_short = model_short[:20] + '...' if len(model_short) > 20 else model_short
            
            for k in [1, 2, 3, 4]:
                val = r.get(f'M_k{k}')
                if pd.notna(val):
                    data_list.append({'k': f'k{k}', 'Model': model_short, 'Score': val})
        
        if not data_list:
            print(f"No valid data to plot for {t}")
            continue
            
        plot_df = pd.DataFrame(data_list)
        
        plt.figure(figsize=(10, 6))
        sns.barplot(data=plot_df, x='k', y='Score', hue='Model')
        plt.title(f"Score Comparison: {t}")
        plt.grid(True, axis='y', alpha=0.3)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

if 'comp_df' in locals() and not comp_df.empty:
    plot_comp_results(comp_df)


--- Comparison for Target: Fire ---
FWI Global Scores: {1: np.float64(0.0116330213361341), 2: np.float64(0.0232660426722682), 3: np.float64(0.0719063733940639), 4: np.float64(0.177215576171875)}
Results for Time not found in all_target_results. Skipping.

--- Comparison for Target: Res ---
FWI Global Scores: {1: np.float64(-0.0056554421917301), 2: np.float64(0.116647947980158), 3: np.float64(0.2403817711551589), 4: np.float64(0.6785278320312501)}
| Target   | Model                                        |   M_k1 |   D_k1 |   M_k2 |   D_k2 |   M_k3 |   D_k3 |   M_k4 |   D_k4 |
|:---------|:---------------------------------------------|-------:|-------:|-------:|-------:|-------:|-------:|-------:|-------:|
| Fire     | GRU_full_full_10_0_all_one_regression_pdegpd |    nan |    nan |    nan |    nan |    nan |    nan |    nan |    nan |
| Res      | GRU_full_full_10_0_all_one_regression_pdegpd |    nan |    nan |    nan |    nan |    nan |    nan |    nan |    nan |
No valid data to plo